In [5]:
%use kandy
import profilelib.*

We are trying to estimate the performance ratio of stdlib functions in Kotlin/Native vs. JVM. For this example, we focus
on `String.substring` method used in this particular scenario. We need to exclude the effect of GC to get the performace
ratio of execution time of the function itself. The following cell describes, how we collect the data. We
end up with a list of execution times.

In [6]:
import java.nio.file.Files
import java.nio.file.Path
import java.nio.file.StandardOpenOption

val kotlincNative = System.getProperty("user.home") + "/.konan/kotlin-native-prebuilt-linux-x86_64-2.2.10/bin/kotlinc-native"
val tempDir = Files.createTempDirectory("knative-perf")

fun runProcess(vararg command: String): String =
    ProcessBuilder(command.toList())
        .directory(tempDir.toFile())
        .start()
        .let { process ->
            process.inputStream.bufferedReader().readText()
                .also {
                    if (process.waitFor() != 0) {
                        process.errorStream.bufferedReader().readText().let(::println)
                        throw Exception("Process failed with code ${process.exitValue()}")
                    }
                }
        }

val source = """
import kotlin.concurrent.Volatile
import kotlin.time.measureTime
import kotlin.time.DurationUnit
import kotlin.time.Duration
import kotlin.native.runtime.GC
import kotlin.random.Random
import kotlin.native.identityHashCode

class Blackhole {
    @Volatile
    var i0: Int = Random.nextInt()
    var i1 = i0 + 1

    @Volatile
    var bh: Blackhole? = null

    @OptIn(kotlin.experimental.ExperimentalNativeApi::class)
    inline fun consume(obj: Any?) {
        // identityHashCode is an intrinsic function
        // resolved into getting an object address, so there will be no call.
        consume(obj.identityHashCode())
    }

    inline fun consume(i: Int) {
        // To ensure that i's value will not be removed by optimizations like dead code elimination,
        // its value is compares with two value i0 and i1, such that i1 = i0 + 1.
        // As long as i0 and i1 are different, the following condition should not ever be met
        // and as the branch following it (note that if it is executed, then NPE will happen).
        // To ensure that at least i0 value will be loaded on every call, it was annotated with Volatile.
        //
        // This approach has one drawback: in general, it should be compiled to a code with two branch instructions,
        // and performance characteristics of a benchmark may not be stable if consumed value is sometimes equal to i0.
        // In practice, there is almost no effect on the measured performance and
        // the difference is within the error margin.
        // However, if it becomes a problem one day, then the condition should be rewritten to something like:
        // if (((i0 xor i) and (i1 xor i)) == -1) { ... }
        // We can't simply compare xor results as it could be optimized to comparison of i0 and i1 and i's evaluation
        // may be sunk into the unreachable branch.
        if ((i0 == i) && (i1 == i)) {
            bh!!.i0 = i
        }
    }
}

val blackhole = Blackhole()
@Volatile
private var string = "abc".repeat(100000)

@kotlin.native.runtime.NativeRuntimeApi
@kotlin.ExperimentalStdlibApi
fun main() {
    var epoch: Long? = null
    repeat(1_000_000) { blackhole.consume(string.substring(10000, 50000)) }
    val iterations = 10_000
    val times = ArrayList<Double?>(iterations)
    repeat(iterations) {
        measureTime {
            repeat(10) {
                blackhole.consume(string.substring(10000, 50000))
            }
        }.let { times.add(it.toDouble(DurationUnit.SECONDS)) }
        epoch = GC.lastGCInfo?.epoch.also {
            if (it != epoch) times.add(null)
        }
    }
    times.forEach { println(it) }
}
"""

val sourceFile = tempDir.resolve("benchmark.kt")
val outputBinary = tempDir.resolve("benchmark.kexe")

Files.writeString(
    sourceFile,
    source,
    StandardOpenOption.CREATE,
)

runProcess(
        kotlincNative,
        sourceFile.toString(),
        "-o", outputBinary.toString(),
)

val times = runProcess(outputBinary.toString())
    .split("\n")
    .mapNotNull { it.takeIf { it.isNotEmpty() } }
    .map { it.toDoubleOrNull() }
    .zipWithNext()
    .partition { it.second == null }
    .let { (x, y) -> mapOf(
        "GC" to x.map { it.first!! },
        "noGC" to y.mapNotNull { it.first },
        )}


For the exclusion of GC, we assume that the execution time distribution is bimodal (i .e. there are two peaks). The
first peak is formed by the executions that didn't trigger GC. The second peak is formed by the executions that
triggered GC. By finding a threshold for the execution time that lies between the two peaks, we can separate the
executions into those that ran GC and those which didn't.

In [7]:
plot_histogram(times)

<head>
 <meta charset="UTF-8">
 <style> html, body { margin: 0; padding: 0; overflow: hidden; } </style>
 <script type="text/javascript" data-lets-plot-script="library" src="https://cdn.jsdelivr.net/gh/JetBrains/lets-plot@v4.5.1/js-package/distr/lets-plot.min.js"></script>
 </head>
 <body>
 <div id="Lg7NXV"></div>
 <script type="text/javascript" data-lets-plot-script="plot">
 
 (function() {
 // ----------
 
 const forceImmediateRender = false;
 const responsive = false;
 
 let sizing = {
 width_mode: "FIXED",
 height_mode: "FIXED",
 width: 600.0, 
 height: 400.0 
 };
 
 const preferredWidth = document.body.dataset.letsPlotPreferredWidth;
 if (preferredWidth !== undefined) {
 sizing = {
 width_mode: 'FIXED',
 height_mode: 'SCALED',
 width: parseFloat(preferredWidth)
 };
 }
 
 const containerDiv = document.getElementById("Lg7NXV");
 let fig = null;
 
 function renderPlot() {
 if (fig === null) {
 const plotSpec = {
"mapping":{
},
"data":{
},
"kind":"plot",
"scales":[{
"aesthetic":"x",
"name":"sample",
"limits":[null,null]
},{
"aesthetic":"x",
"limits":[null,null]
},{
"aesthetic":"y",
"limits":[null,null]
},{
"aesthetic":"fill",
"discrete":true
}],
"layers":[{
"mapping":{
"x":"x",
"y":"count",
"fill":"group",
"group":"&merged_groups"
},
"stat":"identity",
"data":{
"&merged_groups":["GC","GC","GC","GC","GC","GC","GC","GC","GC","GC","GC","GC","GC","GC","GC","GC","GC","GC","GC","GC","noGC","noGC","noGC","noGC","noGC","noGC","noGC","noGC","noGC","noGC","noGC","noGC","noGC","noGC","noGC","noGC","noGC","noGC","noGC","noGC"],
"count":[962.0,832.0,128.0,6.0,5.0,8.0,2.0,3.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,7523.0,525.0,3.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0],
"x":[3.9876375E-5,1.19629125E-4,1.9938187500000002E-4,2.79134625E-4,3.5888737500000003E-4,4.3864012500000006E-4,5.18392875E-4,5.98145625E-4,6.77898375E-4,7.576511250000001E-4,8.37403875E-4,9.17156625E-4,9.96909375E-4,0.0010766621249999999,0.0011564148750000001,0.001236167625,0.0013159203750000002,0.001395673125,0.0014754258750000002,0.001555178625,3.9876375E-5,1.19629125E-4,1.9938187500000002E-4,2.79134625E-4,3.5888737500000003E-4,4.3864012500000006E-4,5.18392875E-4,5.98145625E-4,6.77898375E-4,7.576511250000001E-4,8.37403875E-4,9.17156625E-4,9.96909375E-4,0.0010766621249999999,0.0011564148750000001,0.001236167625,0.0013159203750000002,0.001395673125,0.0014754258750000002,0.001555178625],
"group":["GC","GC","GC","GC","GC","GC","GC","GC","GC","GC","GC","GC","GC","GC","GC","GC","GC","GC","GC","GC","noGC","noGC","noGC","noGC","noGC","noGC","noGC","noGC","noGC","noGC","noGC","noGC","noGC","noGC","noGC","noGC","noGC","noGC","noGC","noGC"]
},
"sampling":"none",
"inherit_aes":false,
"position":"dodge",
"geom":"bar",
"data_meta":{
"series_annotations":[{
"type":"str",
"column":"group"
},{
"type":"float",
"column":"x"
},{
"type":"int",
"column":"count"
},{
"type":"str",
"column":"&merged_groups"
}]
}
}],
"data_meta":{
"series_annotations":[{
"type":"str",
"column":"group"
},{
"type":"str",
"column":"&merged_groups"
}]
},
"spec_id":"5"
};
 fig = LetsPlot.buildPlotFromProcessedSpecs(plotSpec, containerDiv, sizing);
 } else {
 fig.updateView({});
 }
 }
 
 const renderImmediately = 
 forceImmediateRender || (
 sizing.width_mode === 'FIXED' && 
 (sizing.height_mode === 'FIXED' || sizing.height_mode === 'SCALED')
 );
 
 if (renderImmediately) {
 renderPlot();
 }
 
 if (!renderImmediately || responsive) {
 // Set up observer for initial sizing or continuous monitoring
 var observer = new ResizeObserver(function(entries) {
 for (let entry of entries) {
 if (entry.contentBoxSize && 
 entry.contentBoxSize[0].inlineSize > 0) {
 if (!responsive && observer) {
 observer.disconnect();
 observer = null;
 }
 renderPlot();
 if (!responsive) {
 break;
 }
 }
 }
 });
 
 observer.observe(containerDiv);
 }
 
 // ----------
 })();
 
 </script>
 </body>
</html>"> 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 


We then estimate the total time spent in GC by the following formula.

In [8]:
val avgNoGC = times["noGC"]!!.average()
times["GC"]!!.sumOf { it - avgNoGC } / (times["noGC"]!!.sum() + times["GC"]!!.sum())

0.2898553021263625

For measurements that run at least 100 000 iterations and don't print the times throught the execution (but all at once at the end), we estimate that GC is responsible for over 80% of the execution. That is insane. Therefore, we question the results.

* Can GC in K/N be responsible for such a high percentage of execution time? After all, this is very memory heavy benchmark.
* Is there an error in the benchmark setup, collection of the data or the processing?
* Is any one of our assumptions incorrect?
* Is this method for estimating GC time sound?